In [2]:
import datetime
import datajoint as dj

In [3]:
from aeon.dj_pipeline.analysis.block_analysis import *

[2024-04-19 16:45:35,687][INFO]: Connecting jbhagat@aeon-db2:3306
[2024-04-19 16:45:35,702][INFO]: Connected jbhagat@aeon-db2:3306


In [4]:
dj.conn()

DataJoint connection (connected) jbhagat@aeon-db2:3306

In [12]:
# get key from table with multiple subjects
key = (BlockAnalysis.aggr(BlockAnalysis.Subject, subj_count="count(*)") & "subj_count>1").fetch("KEY")[-1]

In [13]:
block_patches = (BlockAnalysis.Patch & key).fetch(as_dict=True)
block_subjects = (BlockAnalysis.Subject & key).fetch(as_dict=True)
subject_names = [s["subject_name"] for s in block_subjects]
# Construct subject position dataframe
subjects_positions_df = pd.concat(

[
    pd.DataFrame(
        {"subject_name": [s["subject_name"]] * len(s["position_timestamps"])}
        | {
            k: s[k]
            for k in (
                "position_timestamps",
                "position_x",
                "position_y",
                "position_likelihood",
            )
        }
    )
    for s in block_subjects
]
)
subjects_positions_df.set_index("position_timestamps", inplace=True)
# Get frame rate of CameraTop
camera_fps = int(
(
    streams.SpinnakerVideoSource * streams.SpinnakerVideoSource.Attribute
    & key
    & 'attribute_name = "SamplingFrequency"'
    & 'spinnaker_video_source_name = "CameraTop"'
    & f'spinnaker_video_source_install_time < "{key["block_start"]}"'
).fetch("attribute_value", order_by="spinnaker_video_source_install_time DESC", limit=1)[0]
)

In [14]:
block_subjects

[{'experiment_name': 'social0.2-aeon3',
  'block_start': datetime.datetime(2024, 2, 22, 4, 3, 4, 7999),
  'subject_name': 'BAA-1104045',
  'weights': array([31.5, 31.5, 31.5, ..., 30.7999992, 30.7999992, 30.7999992],
        dtype=object),
  'weight_timestamps': array(['2024-02-22T05:30:21.940000057', '2024-02-22T05:30:22.000000000',
         '2024-02-22T05:30:22.099999905', ...,
         '2024-02-22T06:03:49.059999943', '2024-02-22T06:03:49.139999866',
         '2024-02-22T06:03:49.219999790'], dtype='datetime64[ns]'),
  'position_x': array([1301.8185, 1301.8226, 1301.8151, ..., 1182.0592, 195.24168,
         1182.0399], dtype=object),
  'position_y': array([512.3161, 512.31726, 512.2335, ..., 757.66547, 560.59674, 757.738],
        dtype=object),
  'position_likelihood': array([0.6524323, 0.65661824, 0.64131397, ..., 0.95852286, 0.958901,
         0.958901], dtype=object),
  'position_timestamps': array(['2024-02-22T04:03:04.019999981', '2024-02-22T04:03:04.079999924',
         '2024

In [24]:
patch1 = (BlockAnalysis.Patch & key & "patch_name = 'Patch1'").fetch1()

In [1]:
subjects_xy

NameError: name 'subjects_xy' is not defined

In [25]:
cum_wheel_dist = pd.Series(
    index=patch["wheel_timestamps"], data=patch["wheel_cumsum_distance_travelled"]
)
# Get distance-to-patch at each pose data timestep
patch_region = (
    acquisition.EpochConfig.ActiveRegion
    & key
    & {"region_name": f"{patch['patch_name']}Region"}
    & f'epoch_start < "{key["block_start"]}"'
).fetch("region_data", order_by="epoch_start DESC", limit=1)[0]
patch_xy = list(zip(*[(int(p["X"]), int(p["Y"])) for p in patch_region["ArrayOfPoint"]]))
patch_center = np.mean(patch_xy[0]).astype(np.uint32), np.mean(patch_xy[1]).astype(np.uint32)
subjects_xy = subjects_positions_df[["position_x", "position_y"]].values
dist_to_patch = np.sqrt(np.sum((subjects_xy - patch_center) ** 2, axis=1).astype(float))
dist_to_patch_df = subjects_positions_df[["subject_name"]].copy()
dist_to_patch_df["dist_to_patch"] = dist_to_patch
# Assign pellets and wheel timestamps to subjects
if len(block_subjects) == 1:
    cum_wheel_dist_dm = cum_wheel_dist.to_frame(name=subject_names[0])
    patch_df_for_pellets_df = pd.DataFrame(
        index=patch["pellet_timestamps"], data={"subject_name": subject_names[0]}
    )
else:
    # Assign id based on which subject was closest to patch at time of event
    # Get distance-to-patch at each wheel ts and pel del ts, organized by subject
    dist_to_patch_wheel_ts_id_df = pd.DataFrame(
        index=cum_wheel_dist.index, columns=subject_names
    )
    dist_to_patch_pel_ts_id_df = pd.DataFrame(
        index=patch["pellet_timestamps"], columns=subject_names
    )
    for subject_name in subject_names:
        # Find closest match between pose_df indices and wheel indices
        if not dist_to_patch_wheel_ts_id_df.empty:
            dist_to_patch_wheel_ts_subj = pd.merge_asof(
                left=dist_to_patch_wheel_ts_id_df[subject_name],
                right=dist_to_patch_df[dist_to_patch_df["subject_name"] == subject_name],
                left_index=True,
                right_index=True,
                direction="forward",
                tolerance=pd.Timedelta("100ms"),
            )
            dist_to_patch_wheel_ts_id_df[subject_name] = dist_to_patch_wheel_ts_subj[
                "dist_to_patch"
            ]
        # Find closest match between pose_df indices and pel indices
        if not dist_to_patch_pel_ts_id_df.empty:
            dist_to_patch_pel_ts_subj = pd.merge_asof(
                left=dist_to_patch_pel_ts_id_df[subject_name],
                right=dist_to_patch_df[dist_to_patch_df["subject_name"] == subject_name],
                left_index=True,
                right_index=True,
                direction="forward",
                tolerance=pd.Timedelta("200ms"),
            )
            dist_to_patch_pel_ts_id_df[subject_name] = dist_to_patch_pel_ts_subj[
                "dist_to_patch"
            ]
    # Get closest subject to patch at each pel del timestep
    patch_df_for_pellets_df = pd.DataFrame(
        index=patch["pellet_timestamps"],
        data={"subject_name": dist_to_patch_pel_ts_id_df.idxmin(axis=1).values},
    )

    # Get closest subject to patch at each wheel timestep
    cum_wheel_dist_subj_df = pd.DataFrame(
        index=cum_wheel_dist.index, columns=subject_names, data=0.0
    )
    closest_subjects = dist_to_patch_wheel_ts_id_df.idxmin(axis=1)
    wheel_dist = cum_wheel_dist.diff().fillna(cum_wheel_dist.iloc[0])
    # Assign wheel dist to closest subject for each wheel timestep
    for subject_name in subject_names:
        subj_idxs = cum_wheel_dist_subj_df[closest_subjects == subject_name].index
        cum_wheel_dist_subj_df.loc[subj_idxs, subject_name] = wheel_dist[subj_idxs]
    cum_wheel_dist_dm = cum_wheel_dist_subj_df.cumsum(axis=0)
    # In Patch Time
    patch_bbox = mpl_path.Path(list(zip(*patch_xy)))
    in_patch = subjects_positions_df.apply(
        lambda row: patch_bbox.contains_point((row["position_x"], row["position_y"])), axis=1
    )

MergeError: Incompatible merge dtype, dtype('O') and dtype('<M8[ns]'), both sides must have numeric dtype